In [3]:
# ============================================================
# V8.0 — PORTABILITY / CONFIGURATION TEST
# ============================================================

import sys

PROJECT_ROOT = (
    "/home/sagemaker-user/adaptive-llm-routing"
)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from app.config.settings import (
    Settings,
    load_v8_config,
    get_enabled_models,
)


print("=" * 100)
print("V8.0 — CONFIGURATION TEST")
print("=" * 100)

Settings.validate()

config = load_v8_config()
models = get_enabled_models()

print()
print("Project:")
print(
    Settings.PROJECT_NAME
)

print()
print("Environment:")
print(
    Settings.ENVIRONMENT
)

print()
print("AWS region:")
print(
    Settings.AWS_REGION
)

print()
print("DynamoDB table:")
print(
    Settings.DYNAMODB_TABLE
)

print()
print("V8 version:")
print(
    config["version"]
)

print()
print("Routing version:")
print(
    config["routing"]["version"]
)

print()
print("Enabled candidate models:")
for logical_name, model_config in models.items():

    print(
        f"  {logical_name}"
    )

    print(
        f"    model_id: "
        f"{model_config['model_id']}"
    )

    print(
        f"    tier: "
        f"{model_config.get('tier')}"
    )

print()
print(
    "Number of enabled models:",
    len(models),
)

print()
print("=" * 100)
print("✅ V8.0 CONFIGURATION TEST PASSED")
print("=" * 100)

V8.0 — CONFIGURATION TEST

Project:
adaptive-llm-routing

Environment:
development

AWS region:
eu-north-1

DynamoDB table:
adaptive-llm-router-state

V8 version:
8.0

Routing version:
v7.7

Enabled candidate models:
  small_model
    model_id: eu.amazon.nova-micro-v1:0
    tier: small
  medium_model
    model_id: eu.amazon.nova-lite-v1:0
    tier: medium
  large_model
    model_id: eu.amazon.nova-pro-v1:0
    tier: large

Number of enabled models: 3

✅ V8.0 CONFIGURATION TEST PASSED


In [2]:
# ============================================================
# V8.0 — IMPORT EXISTING V7 MODEL REGISTRY INTO V8 CONFIG
# ============================================================

import sys
from pathlib import Path
import yaml

PROJECT_ROOT = Path(
    "/home/sagemaker-user/adaptive-llm-routing"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from app.models.v7_bedrock import V7BedrockClient


print("=" * 100)
print("V8.0 — MIGRATING EXISTING V7 MODEL CONFIGURATION")
print("=" * 100)

client = V7BedrockClient(
    region="eu-north-1"
)

logical_models = [
    "small_model",
    "medium_model",
    "large_model",
]

models = {}

for logical_model in logical_models:

    model_id = client.get_model_id(
        logical_model
    )

    models[logical_model] = {
        "model_id": model_id,
        "enabled": True,
        "capabilities": [
            "text"
        ],
        "tier": (
            "small"
            if logical_model == "small_model"
            else "medium"
            if logical_model == "medium_model"
            else "large"
        ),
    }

    print()
    print(
        f"{logical_model} -> {model_id}"
    )


config = {
    "version": "8.0",
    "models": models,
}


config_path = (
    PROJECT_ROOT
    / "config"
    / "models.yaml"
)

with open(
    config_path,
    "w",
    encoding="utf-8",
) as file:

    yaml.safe_dump(
        config,
        file,
        sort_keys=False,
    )


print()
print(
    "Configuration written to:"
)

print(
    config_path
)

print()
print("=" * 100)
print("✅ V8.0 MODEL CONFIGURATION MIGRATION COMPLETE")
print("=" * 100)

V8.0 — MIGRATING EXISTING V7 MODEL CONFIGURATION

small_model -> eu.amazon.nova-micro-v1:0

medium_model -> eu.amazon.nova-lite-v1:0

large_model -> eu.amazon.nova-pro-v1:0

Configuration written to:
/home/sagemaker-user/adaptive-llm-routing/config/models.yaml

✅ V8.0 MODEL CONFIGURATION MIGRATION COMPLETE


In [1]:
# ============================================================
# V8.1 — LIVE BENCHMARK
# ============================================================

import sys

PROJECT_ROOT = (
    "/home/sagemaker-user/adaptive-llm-routing"
)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from app.evaluation.v8_benchmark import (
    V81Benchmark,
)

print("=" * 100)
print("V8.1 — ADAPTIVE ROUTER BENCHMARK")
print("=" * 100)

benchmark = V81Benchmark(
    seed=42
)

results = benchmark.run()

summary = benchmark.summarize(
    results
)

print()
print("=" * 100)
print("V8.1 — FINAL BENCHMARK SUMMARY")
print("=" * 100)

for row in summary:

    print()
    print(
        "Strategy:",
        row["strategy"]
    )

    print(
        "Requests:",
        row["requests"]
    )

    print(
        "Success rate:",
        f"{row['success_rate']:.3f}"
    )

    print(
        "Average quality:",
        f"{row['avg_quality']:.3f}"
    )

    print(
        "Average latency:",
        f"{row['avg_latency_ms']:.2f} ms"
    )

    print(
        "P95 latency:",
        f"{row['p95_latency_ms']:.2f} ms"
    )

    print(
        "Average output tokens:",
        f"{row['avg_output_tokens']:.1f}"
    )

    print(
        "Models used:",
        row["models_used"]
    )

print()
print("=" * 100)
print("✅ V8.1 BENCHMARK COMPLETE")
print("=" * 100)

V8.1 — ADAPTIVE ROUTER BENCHMARK

STRATEGY: strongest

b1 → large_model
success=True latency=2036.51 ms tokens=256 quality=1.000

b2 → large_model
success=True latency=1280.13 ms tokens=256 quality=1.000

b3 → large_model
success=True latency=1736.26 ms tokens=256 quality=1.000

b4 → large_model
success=True latency=1903.49 ms tokens=256 quality=1.000

b5 → large_model
success=True latency=1517.05 ms tokens=256 quality=0.200

b6 → large_model
success=True latency=1527.30 ms tokens=256 quality=1.000

STRATEGY: cheapest

b1 → small_model
success=True latency=1233.52 ms tokens=256 quality=1.000

b2 → small_model
success=True latency=1320.98 ms tokens=256 quality=1.000

b3 → small_model
success=True latency=1160.16 ms tokens=256 quality=0.500

b4 → small_model
success=True latency=1327.00 ms tokens=256 quality=1.000

b5 → small_model
success=True latency=1204.48 ms tokens=256 quality=0.800

b6 → small_model
success=True latency=1440.36 ms tokens=256 quality=1.000

STRATEGY: random

b1 → la

In [2]:
# ============================================================
# V8.2 — EVALUATION + PERSISTENT RESULTS
# ============================================================

import sys

PROJECT_ROOT = (
    "/home/sagemaker-user/adaptive-llm-routing"
)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from app.evaluation.v8_2_evaluator import (
    V82Evaluator,
)

print("=" * 100)
print("V8.2 — EVALUATION + PERSISTENT EXPERIMENT RESULTS")
print("=" * 100)

# ------------------------------------------------------------
# IMPORTANT:
# Reuse the V8.1 `results` object from the previous cell.
# This does NOT run Bedrock again.
# ------------------------------------------------------------

evaluator = V82Evaluator()

report = evaluator.evaluate(
    results
)

print()
print(
    "Benchmark requests:",
    report["benchmark_requests"],
)

print(
    "Strategies evaluated:",
    report["strategies_evaluated"],
)

print()
print(
    "Cost available:",
    report["cost_data_available"],
)

print(
    "Cost note:",
    report["cost_note"],
)

print()
print("=" * 100)
print("V8.2 STRATEGY EVALUATION")
print("=" * 100)

for row in report["strategies"]:

    print()
    print(
        "Strategy:",
        row["strategy"]
    )

    print(
        "Average quality:",
        f"{row['average_quality']:.3f}"
    )

    print(
        "Quality retention vs strongest:",
        (
            f"{row['quality_retention_vs_strongest']:.3f}"
            if row[
                "quality_retention_vs_strongest"
            ] is not None
            else "N/A"
        )
    )

    print(
        "Average latency:",
        f"{row['average_latency_ms']:.2f} ms"
    )

    print(
        "P95 latency:",
        f"{row['p95_latency_ms']:.2f} ms"
    )

    print(
        "Latency reduction vs strongest:",
        (
            f"{row['latency_reduction_vs_strongest']:.3f}"
            if row[
                "latency_reduction_vs_strongest"
            ] is not None
            else "N/A"
        )
    )

    print(
        "Success rate:",
        f"{row['success_rate']:.3f}"
    )

    print(
        "Models:",
        row["models_used"]
    )

    print(
        "Pareto quality/latency:",
        row["pareto_quality_latency"]
    )


# ------------------------------------------------------------
# PERSIST LOCAL RESULTS
# ------------------------------------------------------------

artifacts = evaluator.persist(
    report
)

print()
print("=" * 100)
print("V8.2 ARTIFACTS")
print("=" * 100)

print(
    "JSON:",
    artifacts["json_path"]
)

print(
    "CSV:",
    artifacts["csv_path"]
)

print(
    "S3:",
    (
        artifacts["s3_locations"]
        if artifacts["s3_locations"]
        else "Not configured"
    )
)

print(
    "CloudWatch:",
    (
        "Published"
        if artifacts[
            "cloudwatch_published"
        ]
        else "Not configured"
    )
)

print()
print("=" * 100)
print("✅ V8.2 EVALUATION COMPLETE")
print("=" * 100)

V8.2 — EVALUATION + PERSISTENT EXPERIMENT RESULTS

Benchmark requests: 30
Strategies evaluated: 5

Cost available: False
Cost note: Real inference cost was not available in V8.1. Cost metrics are intentionally omitted rather than estimated.

V8.2 STRATEGY EVALUATION

Strategy: cheapest
Average quality: 0.883
Quality retention vs strongest: 1.019
Average latency: 1281.08 ms
P95 latency: 1412.02 ms
Latency reduction vs strongest: 0.231
Success rate: 1.000
Models: ['small_model']
Pareto quality/latency: True

Strategy: strongest
Average quality: 0.867
Quality retention vs strongest: 1.000
Average latency: 1666.79 ms
P95 latency: 2003.26 ms
Latency reduction vs strongest: 0.000
Success rate: 1.000
Models: ['large_model']
Pareto quality/latency: False

Strategy: random
Average quality: 0.817
Quality retention vs strongest: 0.942
Average latency: 1373.09 ms
P95 latency: 1730.57 ms
Latency reduction vs strongest: 0.176
Success rate: 1.000
Models: ['large_model', 'medium_model', 'small_model']

In [2]:
# ============================================================
# V8.3 — FASTAPI TEST
# ============================================================

import sys

PROJECT_ROOT = (
    "/home/sagemaker-user/adaptive-llm-routing"
)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from app.api_server import app

print("=" * 100)
print("V8.3 — FASTAPI TEST")
print("=" * 100)

print()
print("Application:")
print(app.title)

print()
print("Version:")
print(app.version)

print()
print("Routes:")

for route in app.routes:

    path = getattr(
        route,
        "path",
        None,
    )

    methods = getattr(
        route,
        "methods",
        None,
    )

    if path is not None:

        if methods:
            print(
                f"  {sorted(methods)} {path}"
            )
        else:
            print(
                f"  {path}"
            )

    else:

        print(
            f"  [router object] {type(route).__name__}"
        )

print()
print("=" * 100)
print("✅ V8.3 FASTAPI IMPORT TEST PASSED")
print("=" * 100)

V8.3 — FASTAPI TEST

Application:
Adaptive LLM Router

Version:
8.3.0

Routes:
  ['GET', 'HEAD'] /openapi.json
  ['GET', 'HEAD'] /docs
  ['GET', 'HEAD'] /docs/oauth2-redirect
  ['GET', 'HEAD'] /redoc
  [router object] _IncludedRouter
  ['GET'] /

✅ V8.3 FASTAPI IMPORT TEST PASSED


In [3]:
# ============================================================
# V8.3 — API ENDPOINT TEST
# ============================================================

from fastapi.testclient import TestClient
from app.api_server import app

client = TestClient(app)

print("=" * 100)
print("V8.3 — API ENDPOINT TEST")
print("=" * 100)

# ------------------------------------------------------------
# 1. Health
# ------------------------------------------------------------

response = client.get("/api/health")

print("\n1. /api/health")
print("Status:", response.status_code)
print("Response:", response.json())

assert response.status_code == 200


# ------------------------------------------------------------
# 2. Routing
# ------------------------------------------------------------

route_payload = {
    "prompt": "Explain gradient descent in simple terms."
}

response = client.post(
    "/api/route",
    json=route_payload,
)

print("\n2. /api/route")
print("Status:", response.status_code)
print("Response:", response.json())

assert response.status_code == 200

route_result = response.json()

assert "selected_model" in route_result
assert "decision_type" in route_result


# ------------------------------------------------------------
# 3. Real generation through Bedrock
# ------------------------------------------------------------

generate_payload = {
    "prompt": "What is Python?",
    "max_tokens": 128,
    "temperature": 0.2,
}

response = client.post(
    "/api/generate",
    json=generate_payload,
)

print("\n3. /api/generate")
print("Status:", response.status_code)

generate_result = response.json()

print("Response:")
print(generate_result)

assert response.status_code == 200

assert (
    "response" in generate_result
    or "text" in generate_result
    or "output" in generate_result
)

print("\n" + "=" * 100)
print("✅ V8.3 API ENDPOINT TEST PASSED")
print("=" * 100)

/home/sagemaker-user/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


V8.3 — API ENDPOINT TEST

1. /api/health
Status: 200
Response: {'status': 'ok', 'service': 'adaptive-llm-routing', 'version': 'V8.3', 'aws_region': 'eu-north-1'}

2. /api/route
Status: 200
Response: {'version': 'V8.3', 'selected_model': 'medium_model', 'decision_type': 'EXPLOITATION', 'v75_candidate': 'medium_model', 'v76_best_model': 'medium_model', 'final_score': 0.7956656346749227, 'candidate_models': ['small_model', 'medium_model', 'large_model'], 'cost_available': False}

3. /api/generate
Status: 200
Response:
{'success': True, 'selected_model': 'large_model', 'decision_type': 'EXPLORATION', 'v75_candidate': 'large_model', 'v76_best_model': 'medium_model', 'final_score': 0.23529411764705882, 'latency_ms': 973.4888320090249, 'response': "Python is a high-level, interpreted programming language known for its readability and simplicity. Created by Guido van Rossum and first released in 1991, Python emphasizes code readability with its notable use of significant whitespace. Its design

In [4]:
# ============================================================
# V8.3 — INSTALL API DEPENDENCIES
# ============================================================

import sys
import subprocess

packages = [
    "fastapi>=0.115.0",
    "uvicorn[standard]>=0.30.0",
    "pydantic>=2.0.0",
]

print("Installing V8.3 dependencies...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    *packages,
])

print()
print("✅ V8.3 dependencies installed")

Installing V8.3 dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 164.4 MB/s eta 0:00:00

✅ V8.3 dependencies installed



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
